# Stage 6: network measures

**What this stage does:** computes the measures for every batter and for every network, using the same rules
for all three formats (see `library/lib_05_network_measures.ipynb` and `docs/methods_decisions.md`).

**Reads:** `data/processed/nodes_<format>.csv` and `edges_<format>.csv`
**Writes:** `outputs/tables/node_metrics_<format>.csv`, `network_summary.csv`, `communities_<format>.csv` and
`core_sizes_<format>.csv`

This one takes about a minute, because of the 200 random graphs and the 100 Louvain runs per format.

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_04_network_building.ipynb
%run ./library/lib_05_network_measures.ipynb
stage_log = get_stage_logger("06_measures")

In [2]:
summary_by_format = {}
for format_name in FORMATS:
    file_tag = FORMAT_FILE_TAG[format_name]
    partnership_graph = load_format_graph(format_name)
    node_measure_table = node_measures(partnership_graph)
    write_table(node_measure_table, TABLES_FOLDER / f"node_metrics_{file_tag}.csv")
    summary_by_format[format_name] = network_summary(partnership_graph, node_measure_table)

    # How many batters are in each k-core shell (1-core, 2-core, ...).
    core_sizes = node_measure_table.groupby("core_number").size().rename("players").reset_index()
    core_sizes.insert(0, "format", format_name)
    write_table(core_sizes, TABLES_FOLDER / f"core_sizes_{file_tag}.csv")

    # Who is in each Louvain community, strongest batter first.
    community_table = (node_measure_table.sort_values(["community", "strength"], ascending=[True, False])
                       .groupby("community")
                       .agg(players=("player_id", "size"), total_node_strength=("strength", "sum"),
                            members=("player_name", lambda names: "; ".join(names)))
                       .reset_index())
    community_table.insert(0, "format", format_name)
    write_table(community_table, TABLES_FOLDER / f"communities_{file_tag}.csv")
    stage_log.info("%s: %s", format_name, {key: (round(value, 4) if isinstance(value, float) else value)
                                           for key, value in summary_by_format[format_name].items()})

network_summary_table = pd.DataFrame(summary_by_format).rename_axis("measure").reset_index()
write_table(network_summary_table, TABLES_FOLDER / "network_summary.csv")

2026-11-19 18:05:13,352 INFO partnerships.06_measures: Test: {'nodes': 51, 'edges': 335, 'density': 0.2627, 'degree_mean': 13.1373, 'components': 1, 'main_component_nodes': 51, 'component_ratio': 0.0, 'connectedness': 1.0, 'compactness': 0.6064, 'degree_median': 12.0, 'degree_min': 1, 'degree_max': 38, 'degree_sd': 8.7316, 'degree_centralisation': 0.5176, 'strength_mean': 1689.4118, 'strength_median': 623.0, 'strength_max': 15303, 'edge_weight_total': 43080, 'edge_weight_mean': 128.597, 'edge_weight_median': 42.0, 'edge_weight_max': 2697, 'edges_single_partnership_share': 0.391, 'top10pct_edges_weight_share': 0.6273, 'avg_clustering': 0.6164, 'transitivity': 0.5174, 'mean_geodesic_distance': 1.8886, 'diameter': 4, 'mean_redundancy': 6.8464, 'max_core_k': 11, 'max_core_nodes': 20, 'louvain_communities': 5, 'louvain_modularity': 0.2241, 'louvain_sizes': '24|14|7|4|2', 'median_innings_batted': 18, 'C_random': 0.2616, 'L_random': 1.7575, 'C_ratio': 2.3562, 'L_ratio': 1.0746, 'small_world_s

2026-11-19 18:05:13,865 INFO partnerships.06_measures: ODI: {'nodes': 50, 'edges': 262, 'density': 0.2139, 'degree_mean': 10.48, 'components': 1, 'main_component_nodes': 50, 'component_ratio': 0.0, 'connectedness': 1.0, 'compactness': 0.5656, 'degree_median': 9.0, 'degree_min': 1, 'degree_max': 24, 'degree_sd': 6.8071, 'degree_centralisation': 0.2874, 'strength_mean': 917.16, 'strength_median': 427.5, 'strength_max': 4837, 'edge_weight_total': 22929, 'edge_weight_mean': 87.5153, 'edge_weight_median': 40.0, 'edge_weight_max': 953, 'edges_single_partnership_share': 0.5076, 'top10pct_edges_weight_share': 0.487, 'avg_clustering': 0.5266, 'transitivity': 0.4667, 'mean_geodesic_distance': 2.0433, 'diameter': 4, 'mean_redundancy': 4.7342, 'max_core_k': 9, 'max_core_nodes': 23, 'louvain_communities': 4, 'louvain_modularity': 0.2892, 'louvain_sizes': '19|15|8|8', 'median_innings_batted': 11.0, 'C_random': 0.2142, 'L_random': 1.8663, 'C_ratio': 2.4588, 'L_ratio': 1.0948, 'small_world_sigma': 2.2

2026-11-19 18:05:14,294 INFO partnerships.06_measures: T20I: {'nodes': 44, 'edges': 222, 'density': 0.2347, 'degree_mean': 10.0909, 'components': 1, 'main_component_nodes': 44, 'component_ratio': 0.0, 'connectedness': 1.0, 'compactness': 0.5856, 'degree_median': 10.0, 'degree_min': 1, 'degree_max': 27, 'degree_sd': 7.021, 'degree_centralisation': 0.412, 'strength_mean': 862.4545, 'strength_median': 248.5, 'strength_max': 6281, 'edge_weight_total': 18974, 'edge_weight_mean': 85.4685, 'edge_weight_median': 30.5, 'edge_weight_max': 1397, 'edges_single_partnership_share': 0.4054, 'top10pct_edges_weight_share': 0.5018, 'avg_clustering': 0.5494, 'transitivity': 0.5019, 'mean_geodesic_distance': 1.9588, 'diameter': 4, 'mean_redundancy': 4.9533, 'max_core_k': 8, 'max_core_nodes': 22, 'louvain_communities': 4, 'louvain_modularity': 0.1883, 'louvain_sizes': '23|10|10|1', 'median_innings_batted': 9.5, 'C_random': 0.2362, 'L_random': 1.8338, 'C_ratio': 2.3256, 'L_ratio': 1.0681, 'small_world_sigma

### The network-level results side by side

In [3]:
network_summary_table

,measure,Test,ODI,T20I
0,nodes,51,50,44
1,edges,335,262,222
2,density,0.262745,0.213878,0.234672
3,degree_mean,13.137255,10.48,10.090909
4,components,1,1,1
5,main_component_nodes,51,50,44
6,component_ratio,0.0,0.0,0.0
7,connectedness,1.0,1.0,1.0
8,compactness,0.606405,0.565646,0.585624
9,degree_median,12.0,9.0,10.0
